In [ ]:
# Install the Kaggle CLI so Colab can authenticate and download the dataset
!pip install -q kaggle

import os

# Provide your Kaggle username and key here before running this cell in Colab
os.environ['KAGGLE_USERNAME'] = "INSERT_USERNAME_HERE"
os.environ['KAGGLE_KEY'] = "INSERT_KEY_HERE"

# Download the PlantVillage dataset and unpack it into a local working folder
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d dataset
print("Dataset successfully downloaded and unzipped!")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:20<00:00, 108MB/s]

Dataset successfully downloaded and unzipped!


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms as T
from torch.utils.data import DataLoader, random_split

# Resize, augment, and normalize images so the model sees more varied training data
image_transformation = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# PlantVillage stores the RGB images in the 'color' directory
data_dir = 'dataset/plantvillage dataset/color'
full_dataset = datasets.ImageFolder(root=data_dir, transform=image_transformation)

# 80/20 Train-Test Split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

# Create loaders so batches can be streamed efficiently through the model
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
class_names = full_dataset.classes

print(f"Success! Found {len(class_names)} classes: {class_names[:3]}...")

Success! Found 38 classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']...


In [ ]:
# Use the Colab GPU when available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Start from a pretrained MobileNetV2 backbone for transfer learning
model = models.mobilenet_v2(weights='DEFAULT')

# Freeze the backbone first so only the classification head learns new classes
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier output layer with one neuron per plant disease class
num_classes = len(class_names)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

print(f"Model loaded and sent to {device}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 141MB/s]


Model loaded and sent to cuda


In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Cross-entropy is the standard loss for multi-class classification
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# Train one full epoch over the current data loader and return average loss
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

# Evaluate the model on the validation set and return accuracy
def validate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# First train only the new classifier head so the model adapts quickly (3 Epochs)
print("Starting Warm-up Phase...")
for epoch in range(3):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    accuracy = validate(model, val_loader)
    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {accuracy:.2f}%")
    scheduler.step(accuracy)


# Unfreeze the deepest feature blocks for a final fine-tuning pass
print("\nUnfreezing top layers for fine-tuning...")
for param in model.features[14:].parameters():
    param.requires_grad = True

# Rebuild the optimizer so it updates both the classifier head and unfrozen layers
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Fine-tune for 7 Epochs to improve accuracy without overtraining
for epoch in range(7): # 7 more epochs for a total of 10
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    accuracy = validate(model, val_loader)
    print(f"Fine-tune Epoch {epoch+1} | Loss: {train_loss:.4f} | Val Acc: {accuracy:.2f}%")
    scheduler.step(accuracy)

Starting Warm-up Phase...


Training: 100%|██████████| 1358/1358 [04:25<00:00,  5.11it/s]


Epoch 1 | Loss: 0.6829 | Val Acc: 93.57%


Training: 100%|██████████| 1358/1358 [04:20<00:00,  5.22it/s]


Epoch 2 | Loss: 0.2753 | Val Acc: 94.72%


...

Fine-tune Epoch 5 | Loss: 0.0266 | Val Acc: 99.08%


Training: 100%|██████████| 1358/1358 [04:37<00:00,  4.89it/s]


Fine-tune Epoch 6 | Loss: 0.0224 | Val Acc: 99.15%


Training: 100%|██████████| 1358/1358 [04:39<00:00,  4.86it/s]


Fine-tune Epoch 7 | Loss: 0.0208 | Val Acc: 99.23%


In [ ]:
import torch.onnx

# 1. Ensure the model is in evaluation mode
model.eval()

# 2. Define the filepath for your exported model
onnx_file_path = "plant_model.onnx"

# 3. Create a dummy input matching your training transformations (1, 3, 224, 224)
# This acts as a template for the ONNX tracer
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# 4. Export the model
torch.onnx.export(
    model, 
    dummy_input, 
    onnx_file_path, 
    export_params=True,      # Store weights within the file
    opset_version=12,        # Standard version for Flutter/Mobile compatibility
    do_constant_folding=True, # Optimization: simplify constant math
    input_names=['input'],   # Name of the input node for the Flutter wrapper
    output_names=['output']  # Name of the output node
)

print(f"Success! Model exported to {onnx_file_path}")